<!-- field-paper-role-banner -->
> 🟩 **정답·해설본** — 같은 셀 위치의 실행 가능한 참고 구현입니다. 복사하기 전에 수식과 tensor shape를 설명하세요.

# 자기지도 학습 · 멀티모달 · 01. MoCo: momentum dictionary와 queue

## Momentum Contrast for Unsupervised Visual Representation Learning

- **원 논문:** [Momentum Contrast for Unsupervised Visual Representation Learning](https://arxiv.org/abs/1911.05722)
- **저자 / 발표:** Kaiming He, Haoqi Fan, Yuxin Wu, Saining Xie, Ross Girshick · CVPR (2020)
- **난이도 / 예상 시간:** 중급 · 약 70분
- **선수 지식:** contrastive loss, EMA, queue
- **로컬 학습 데이터:** `data/field_curriculum/multimodal_pairs.npz`

이 자료는 핵심 메커니즘을 확인하는 **하드웨어 인식 교육용 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하고, NumPy·환경 simulation 같은
CPU 연산은 그대로 유지합니다. 원 논문의 전체 데이터·모델·학습 예산·최종 benchmark를
재현했다는 뜻은 아닙니다.

## 재현 범위

**보존하는 핵심:** query/key encoder, momentum update, FIFO negative queue를 갖는 MoCo 한 학습 단계를 구현합니다.

**축소하거나 생략한 부분:** ImageNet ResNet-50·65K queue 대신 8×8 이미지·64개 queue를 사용합니다.

완료 기준은 정답과 문자가 같은 코드가 아니라, TODO 구현·불변식 assertion·작은 metric 또는
시각화·마지막 해석을 모두 마치는 것입니다.

## 원 논문 ↔ 실습 지도

        절·수식·그림 번호는 위 primary source를 기준으로 합니다. 판본에 따라 페이지가 달라질 수
        있으므로 절 제목과 식 번호를 함께 확인하세요.

        | 원 논문의 위치 | 대응 코드·실험 | 확인할 증거 |
        |---|---|---|
        | Algorithm 1, lines 1–3 및 §3.1 | 두 stochastic view→query/key encoder | view 차이·unit-norm embedding shape |
| §3.1, Eq. (1) | query-positive/queue-negative InfoNCE | logit shape·target 0 |
| §3.2, Eq. (2) | key encoder momentum update | EMA 수치 |
| §3.2, dictionary as a queue | enqueue/dequeue | 고정 길이·unit norm |
| Algorithm 1 | stop-gradient key branch 학습 | loss·positive similarity |

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
from copy import deepcopy
from pathlib import Path
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(17)
np.random.seed(17)
torch.manual_seed(17)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(17)

DATA_PATH = Path("data/field_curriculum/multimodal_pairs.npz")
assert DATA_PATH.exists(), f"먼저 로컬 학습 데이터를 준비하세요: {DATA_PATH}"
raw = np.load(DATA_PATH, allow_pickle=False)
images = torch.tensor(raw["images"], dtype=torch.float32)
tokens = torch.tensor(raw["tokens"], dtype=torch.long)
class_ids = torch.tensor(raw["class_ids"], dtype=torch.long)
sequences = torch.tensor(raw["sequences"], dtype=torch.float32)
train_idx = torch.tensor(raw["train_idx"], dtype=torch.long)
test_idx = torch.tensor(raw["test_idx"], dtype=torch.long)
assert len(images) == len(tokens) == len(class_ids) == len(sequences)
assert images.shape[1:] == (1, 8, 8) and tokens.ndim == 2 and sequences.ndim == 3
print(
    f"pairs={len(images)}, classes={class_ids.unique().numel()}, "
    f"image={tuple(images.shape[1:])}, tokens={tokens.shape[1]}"
)
print(ACCELERATOR.summary())
print("dataset/index preprocessing stays on CPU, while neural training uses DEVICE")

## 포트폴리오 해설: 표현, 목적함수, 업데이트

$$\mathcal L_q=-\log\frac{\exp(q\cdot k_+/\tau)}{\exp(q\cdot k_+/\tau)+\sum_{k_-}\exp(q
\cdot k_-/\tau)}$$

- **기호 정의:** q는 query, k+는 momentum positive, k-는 queue negative, τ는 온도입니다.
- **수식의 역할:** EMA key encoder와 FIFO queue로 크고 일관된 dictionary를 유지합니다.
- **구현 이유:** representation 붕괴·modality 누설·잘못된 positive 연결을
  코드에서 확인할 수 있도록 핵심 수식을 view 준비와 loss 경계로 분리했습니다.
- **Task/코드 대응:** Task P의 `prepare_views`, 논문 전용 `nn.Module`의
  `forward`, `compute_loss`, `training_step`이 §3.1, Eq. (1)을 구현합니다.
- **입출력 shape:** 두 image view [B, 1, 8, 8] → q/k [B, D] → logit [B, 1+Q]
- **평가:** contrastive loss, positive-vs-queue cosine, queue 길이와 norm
- **원문 대비 한계:** ImageNet ResNet-50·65K queue 대신 8×8 이미지·64개 queue를 사용합니다.

구현은 `로컬 데이터 통계 → view/modality 준비 → model → objective/update → 평가`
순서입니다. 이 순서는 데이터 누설과 stop-gradient 경계를 코드에서 찾기 쉽게 합니다.

### 구현 단계 2 · 핵심 연산·모델

- **원문 위치:** Algorithm 1, lines 1–3 및 §3.1
- **이 셀의 책임:** 두 stochastic view→query/key encoder
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 차이·unit-norm embedding shape. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def image_view(x: Tensor, scale: float):
    return (x * scale + 0.035 * torch.randn_like(x)).clamp(0, 1)


class TinyEncoder(nn.Module):
    def __init__(self, dim=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=1)

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** §3.2, Eq. (2)
- **이 셀의 책임:** key encoder momentum update
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** EMA 수치. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
@torch.no_grad()
def momentum_update(query: nn.Module, key: nn.Module, momentum: float):
    for q, k in zip(query.parameters(), key.parameters()):
        k.data.mul_(momentum).add_(q.data, alpha=1 - momentum)


query_encoder = TinyEncoder()
key_encoder = deepcopy(query_encoder)
before = next(key_encoder.parameters()).clone()
next(query_encoder.parameters()).data.add_(0.1)
momentum_update(query_encoder, key_encoder, 0.9)
assert torch.allclose(
    next(key_encoder.parameters()),
    0.9 * before + 0.1 * next(query_encoder.parameters()),
    atol=1e-6,
)

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** §3.1, Eq. (1)
- **이 셀의 책임:** query-positive/queue-negative InfoNCE
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** logit shape·target 0. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def moco_objective(q: Tensor, k: Tensor, queue: Tensor, temperature=0.2):
    positive = (q * k).sum(1, keepdim=True)
    negative = q @ queue.T
    logits = torch.cat([positive, negative], 1) / temperature
    return F.cross_entropy(
        logits, torch.zeros(len(q), dtype=torch.long, device=logits.device)
    ), logits


queue = F.normalize(torch.randn(64, 16), dim=1)
sample = images[:24]
q = query_encoder(image_view(sample, 0.9))
k = key_encoder(image_view(sample, 1.1)).detach()
loss, logits = moco_objective(q, k, queue)
assert logits.shape == (24, 65) and torch.isfinite(loss)

### 구현 단계 5 · 평가·완료 증거

- **원문 위치:** Algorithm 1, lines 1–3 및 §3.1 / §3.1, Eq. (1) / §3.2, Eq. (2) / §3.2, dictionary as a queue / Algorithm 1
- **이 셀의 책임:** 두 stochastic view→query/key encoder / query-positive/queue-negative InfoNCE / key encoder momentum update / enqueue/dequeue / stop-gradient key branch 학습
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** view 차이·unit-norm embedding shape / logit shape·target 0 / EMA 수치 / 고정 길이·unit norm / loss·positive similarity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
query_encoder, key_encoder, queue = ACCELERATOR.move(query_encoder, key_encoder, queue)
optimizer = torch.optim.Adam(query_encoder.parameters(), lr=0.02)
losses = []
batch = ACCELERATOR.move(images[train_idx[:96]])
for step in range(100):
    ids = torch.arange(step * 24, (step + 1) * 24, device=batch.device) % len(batch)
    x = batch[ids]
    q = query_encoder(image_view(x, 0.9))
    with torch.no_grad():
        momentum_update(query_encoder, key_encoder, 0.97)
        k = key_encoder(image_view(x, 1.1))
    loss, _ = moco_objective(q, k, queue)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    losses.append(float(loss.detach().cpu()))
    queue = torch.cat([k, queue], 0)[:64].detach()
with torch.no_grad():
    test_images = ACCELERATOR.move(images[test_idx])
    q = query_encoder(test_images)
    k = key_encoder(test_images)
    positive = float((q * k).sum(1).mean())
    negative = float((q @ queue.T).mean())
assert (
    queue.shape == (64, 16)
    and torch.allclose(
        queue.norm(dim=1), torch.ones(64, device=queue.device), atol=1e-5
    )
    and positive > negative
)
plt.plot(losses)
plt.title("MoCo InfoNCE")
plt.xlabel("step")
plt.show()
print({"positive": round(positive, 3), "queue_negative": round(negative, 3)})

### Task P1 · modality/view 준비와 핵심 모델

image·text·sequence의 shape를 먼저 고정하고 augmentation 또는 modality
경계를 명시한 뒤 논문 전용 `nn.Module`을 구성합니다.

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** Algorithm 1, lines 1–3 및 §3.1 / §3.1, Eq. (1) / §3.2, Eq. (2) / §3.2, dictionary as a queue / Algorithm 1
- **이 셀의 책임:** 두 stochastic view→query/key encoder / query-positive/queue-negative InfoNCE / key encoder momentum update / enqueue/dequeue / stop-gradient key branch 학습
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** view 차이·unit-norm embedding shape / logit shape·target 0 / EMA 수치 / 고정 길이·unit norm / loss·positive similarity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class PortfolioConfig:
    hidden_dim: int = 16
    temperature: float = 0.2
    momentum: float = 0.9
    learning_rate: float = 0.02
    steps: int = 14


def prepare_views(batch: Tensor) -> tuple[Tensor, Tensor]:
    first = (batch + 0.03 * torch.randn_like(batch)).clamp(0.0, 1.0)
    second = torch.flip(batch, dims=[3])
    return first, second


class MocoPortfolioModel(nn.Module):
    def __init__(self, config: PortfolioConfig):
        super().__init__()
        self.config = config
        self.online = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, config.hidden_dim),
        )
        self.target = deepcopy(self.online)
        for parameter in self.target.parameters():
            parameter.requires_grad_(False)

        self.register_buffer(
            "queue", F.normalize(torch.randn(48, config.hidden_dim), dim=1)
        )
        self.register_buffer("queue_pointer", torch.zeros((), dtype=torch.long))

    def forward(self, first: Tensor, second: Tensor) -> tuple[Tensor, Tensor]:
        online = F.normalize(self.online(first), dim=1)
        with torch.no_grad():
            target = F.normalize(self.target(second), dim=1)
        return online, target

    def compute_loss(self, online: Tensor, target: Tensor) -> Tensor:

        positive = (online * target).sum(dim=1, keepdim=True)
        negative = online @ self.queue.T
        logits = torch.cat([positive, negative], dim=1) / self.config.temperature
        labels = torch.zeros(len(logits), dtype=torch.long, device=logits.device)
        return F.cross_entropy(logits, labels)

    def update_target(self) -> None:
        with torch.no_grad():
            for target, online in zip(
                self.target.parameters(),
                self.online.parameters(),
                strict=True,
            ):
                target.mul_(self.config.momentum)
                target.add_(online, alpha=1.0 - self.config.momentum)

    @torch.no_grad()
    def enqueue(self, keys: Tensor) -> None:
        keys = keys.detach()
        capacity = len(self.queue)
        if len(keys) >= capacity:
            self.queue.copy_(keys[-capacity:])
            self.queue_pointer.zero_()
            return
        pointer = int(self.queue_pointer)
        first_count = min(capacity - pointer, len(keys))
        self.queue[pointer : pointer + first_count] = keys[:first_count]
        remaining = len(keys) - first_count
        if remaining:
            self.queue[:remaining] = keys[first_count:]
        self.queue_pointer.fill_((pointer + len(keys)) % capacity)

    def training_step(self, batch: Tensor) -> tuple[Tensor, Tensor]:
        first, second = prepare_views(batch)
        online, target = self(first, second)
        return self.compute_loss(online, target), target

### Task P2 · objective, stop-gradient, update

positive/negative, online/target 또는 masked/visible 경계를 확인하면서
`fit_portfolio_model`의 gradient와 EMA update 순서를 추적합니다.

### 구현 단계 7 · 목적함수·학습 update

- **원문 위치:** §3.1, Eq. (1) / §3.2, Eq. (2) / §3.2, dictionary as a queue / Algorithm 1
- **이 셀의 책임:** query-positive/queue-negative InfoNCE / key encoder momentum update / enqueue/dequeue / stop-gradient key branch 학습
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** logit shape·target 0 / EMA 수치 / 고정 길이·unit norm / loss·positive similarity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def fit_portfolio_model(model: MocoPortfolioModel, batch: Tensor) -> list[float]:
    trainable = [
        parameter for parameter in model.parameters() if parameter.requires_grad
    ]
    optimizer = torch.optim.Adam(trainable, lr=model.config.learning_rate)
    history = []
    initial_queue = model.queue.clone()
    initial_pointer = int(model.queue_pointer)
    for _ in range(model.config.steps):
        loss, keys = model.training_step(batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        model.update_target()
        model.enqueue(keys)
        history.append(float(loss.detach().cpu()))
    assert not torch.allclose(initial_queue, model.queue)
    assert int(model.queue_pointer) != initial_pointer
    return history

### Task P3 · 표현 품질 평가

학습 loss와 retrieval·reconstruction·conditioning 지표를 분리해 계산합니다.
작은 로컬 pair의 수치는 원 논문의 downstream benchmark와 직접 비교하지 않습니다.

### 구현 단계 8 · 평가·완료 증거

- **원문 위치:** Algorithm 1
- **이 셀의 책임:** stop-gradient key branch 학습
- **Tensor shape 계약:** 두 view [B,C,H,W] → q/k [B,D] → queue logits [B,1+Q]
- **구현 이유:** 학습 숫자 하나가 아니라 논문 주장에 대응하는 반증 가능 증거를 남깁니다.
- **완료 증거:** loss·positive similarity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
def evaluate_portfolio_model(
    model: MocoPortfolioModel,
    batch: Tensor,
) -> dict[str, float]:
    with torch.no_grad():
        first, second = prepare_views(batch)
        online, target = model(first, second)
        agreement = (online * target).sum(dim=1).mean()
        spread = online.std(dim=0).mean()
    return {
        "view_agreement": float(agreement.cpu()),
        "feature_spread": float(spread.cpu()),
    }


portfolio_train_index = train_idx[:32]
portfolio_test_index = test_idx
assert not torch.isin(portfolio_train_index, portfolio_test_index).any()
portfolio_train_images, portfolio_test_images = ACCELERATOR.move(
    images[portfolio_train_index],
    images[portfolio_test_index],
)
portfolio_model = ACCELERATOR.move(MocoPortfolioModel(PortfolioConfig()))
portfolio_history = fit_portfolio_model(portfolio_model, portfolio_train_images)
portfolio_metrics = evaluate_portfolio_model(portfolio_model, portfolio_test_images)
assert np.isfinite(portfolio_history).all()
print(portfolio_metrics)

## 정답 해석 가이드

핵심은 최종 숫자를 외우는 것이 아니라 **원문 위치 → 수식 → 코드 → 반증 가능한 증거**를
연결하는 것입니다. 이 노트북은 다음 원 규모 조건을 재현하지 않았습니다:

> ImageNet ResNet-50·65K queue 대신 8×8 이미지·64개 queue를 사용합니다.

다음 실험에서는 데이터 크기, 모델 폭, 학습 step 중 하나만 바꾸고 동일 seed에서 비교하세요.